In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent  
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import optuna
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from optuna.samplers import TPESampler
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from tqdm.notebook import tqdm
from auxiliary_elements import to_num_nonbin, to_delete, to_category

In [ ]:
binary_column = [
                    "gender", "SeniorCitizen", "Partner", 
                    "Dependents", "PhoneService", "PaperlessBilling",
                ]
to_binary_column = [
                    "MultipleLines", "OnlineSecurity", "OnlineBackup",
                    "DeviceProtection", "TechSupport", "StreamingTV",
                    "StreamingMovies",
                ]
cat_column = [
                "remainder__InternetService", "remainder__Contract", 
                "remainder__PaymentMethod",
            ]

def create_pipeline(model):
    preprocess = ColumnTransformer(transformers=[
    ("to_num_bin", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), binary_column),
    ("to_num_nbin", FunctionTransformer(to_num_nonbin), to_binary_column),
    ], remainder='passthrough')

    preprocess.set_output(transform="pandas")

    pipe = Pipeline([
    ("to_delete", FunctionTransformer(to_delete)),
    ("preprocess", preprocess),
    ("to_category", FunctionTransformer(to_category, kw_args={"col": cat_column})),
    ("model", model),
    ])

    return pipe

pipeline_cat_boost = create_pipeline(CatBoostClassifier(cat_features=tuple(cat_column), verbose=0, auto_class_weights="Balanced", random_seed=42))
pipeline_lgbm = create_pipeline(LGBMClassifier(categorical_feature=cat_column, class_weight="balanced", random_state=42, verbose=-1))


In [ ]:
df = pd.read_csv('../Data/Telco-Customer-Churn.csv')

In [ ]:
X = df.drop(["Churn"], axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.12, random_state=42)

In [ ]:
def objective(trial):
    params = {
        "model__n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "model__max_depth" : trial.suggest_int("max_depth", 2, 10),
        "model__learning_rate" : trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "model__subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "model__l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10)
    }

    pipeline_cat_boost.set_params(**params)
    score = cross_val_score(pipeline_cat_boost, X_train, y_train, cv=5, scoring="accuracy").mean()
    return score

pbar = tqdm(total=25, desc='Optuna tuning')

def tqdm_call_back(study, trial):
    pbar.update(1)
    pbar.set_postfix(best_score=f"{study.best_value:.4f}")

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=25, callbacks=[tqdm_call_back])


In [ ]:
params={
        "model__n_estimators": study.best_params["n_estimators"],
        "model__max_depth" :study.best_params["max_depth"],
        "model__learning_rate" : study.best_params["learning_rate"],
        "model__subsample": study.best_params["subsample"],
        "model__l2_leaf_reg": study.best_params["l2_leaf_reg"],
    }

In [ ]:
model_cb = pipeline_cat_boost.set_params(**params)

In [ ]:
model_cb.fit(X_train, y_train)

In [ ]:
ConfusionMatrixDisplay.from_estimator(model_cb, X_test, y_test)

In [ ]:
y_pred = model_cb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
with open("../model/model_cb.joblib", "wb") as f:
    joblib.dump(model_cb, f)

In [ ]:
def objective(trial):
    params = {
        "model__n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "model__num_leaves" : trial.suggest_int("num_leaves", 15, 200),
        "model__learning_rate" : trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "model__subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "model__max_depth": trial.suggest_int("max_depth", 2, 10)
    }

    pipeline_lgbm.set_params(**params)
    score = cross_val_score(pipeline_lgbm, X_train, y_train, cv=5, scoring="accuracy").mean()
    return score

pbar = tqdm(total=25, desc='Optuna tuning')

def tqdm_call_back(study, trial):
    pbar.update(1)
    pbar.set_postfix(best_score=f"{study.best_value:.4f}")

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=25, callbacks=[tqdm_call_back])

In [ ]:
params = {
    "model__n_estimators" : study.best_params["n_estimators"],
    "model__num_leaves" : study.best_params["num_leaves"],
    "model__learning_rate": study.best_params["learning_rate"],
    "model__subsample": study.best_params["subsample"],
    "model__max_depth": study.best_params["max_depth"],
}

In [ ]:
model_lg = pipeline_lgbm.set_params(**params)

In [ ]:
model_lg.fit(X_train, y_train)

In [ ]:
ConfusionMatrixDisplay.from_estimator(model_lg, X_test, y_test)

In [ ]:
y_pred = model_lg.predict(X_test)
print(classification_report(y_test, y_pred))